In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import sys
import pathlib
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# Setup path
NOTEBOOK_DIR = pathlib.Path(os.getcwd())
REPO_ROOT = NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import cupy as cp
from larndsim.far_field.voxelization import gpu_voxelize, voxel_id_to_coordinates

%matplotlib widget

print("Imports successful")

In [ ]:
# Create a simple track segment that crosses multiple voxels
# Segment from (0, 0, 0) to (10, 8, 6) with 1000 electrons

dtype = np.dtype([
    ('x_start', 'f4'), ('y_start', 'f4'), ('z_start', 'f4'),
    ('x_end', 'f4'), ('y_end', 'f4'), ('z_end', 'f4'),
    ('n_electrons', 'f4')
])

track = np.array([
    (0.0, 0.0, 0.0, 10.0, 8.0, 6.0, 1000.0),
    # (0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1000.0)
], dtype=dtype)

print("Track segment:")
print(f"  Start: ({track['x_start'][0]:.1f}, {track['y_start'][0]:.1f}, {track['z_start'][0]:.1f}) cm")
print(f"  End:   ({track['x_end'][0]:.1f}, {track['y_end'][0]:.1f}, {track['z_end'][0]:.1f}) cm")
print(f"  Charge: {track['n_electrons'][0]:.0f} electrons")
print(f"  Length: {np.sqrt((track['x_end'][0]-track['x_start'][0])**2 + (track['y_end'][0]-track['y_start'][0])**2 + (track['z_end'][0]-track['z_start'][0])**2):.2f} cm")

In [ ]:
# Define voxel parameters
voxel_size = (2.0, 2.0, 0.5)  # 2cm x 2cm x 2cm voxels
tpc_borders = np.array([[[-1, 12], [-1, 10], [-1, 8]]], dtype=np.float32)

print("Voxelization parameters:")
print(f"  Voxel size: {voxel_size[0]} x {voxel_size[1]} x {voxel_size[2]} cm")
print(f"  TPC bounds: x=[{tpc_borders[0,0,0]:.0f}, {tpc_borders[0,0,1]:.0f}], y=[{tpc_borders[0,1,0]:.0f}, {tpc_borders[0,1,1]:.0f}], z=[{tpc_borders[0,2,0]:.0f}, {tpc_borders[0,2,1]:.0f}] cm")

In [ ]:
# Perform voxelization
vox_idx, vox_charge, grid_shape, vsize, bounds = gpu_voxelize(
    track, tpc_borders=tpc_borders, voxel_size=voxel_size
)

vox_charge_np = cp.asnumpy(vox_charge)
print("\nVoxelization results:")
print(f"  Grid shape: {grid_shape}")
print(f"  Total voxels in grid: {grid_shape[0] * grid_shape[1] * grid_shape[2]}")
print(f"  Occupied voxels: {len(vox_idx)}")
print(f"  Occupancy: {100*len(vox_idx)/(grid_shape[0]*grid_shape[1]*grid_shape[2]):.1f}%")

# Get voxel coordinates (returns NumPy arrays)
x_centers, y_centers, z_centers = voxel_id_to_coordinates(vox_idx, grid_shape, vsize, bounds)

print(f"\nCharge conservation:")
print(f"  Input: {track['n_electrons'][0]:.1f} electrons")
print(f"  Output: {vox_charge_np.sum():.1f} electrons")
print(f"  Difference: {abs(vox_charge_np.sum() - track['n_electrons'][0]):.2e} electrons")

In [ ]:
# Helper function to draw a voxel as a wireframe cube
def draw_voxel(ax, x, y, z, dx, dy, dz, color='blue', alpha=0.2, linewidth=0.5):
    """Draw a voxel as a wireframe box."""
    # Define the 8 vertices of the box
    vertices = [
        [x-dx/2, y-dy/2, z-dz/2],
        [x+dx/2, y-dy/2, z-dz/2],
        [x+dx/2, y+dy/2, z-dz/2],
        [x-dx/2, y+dy/2, z-dz/2],
        [x-dx/2, y-dy/2, z+dz/2],
        [x+dx/2, y-dy/2, z+dz/2],
        [x+dx/2, y+dy/2, z+dz/2],
        [x-dx/2, y+dy/2, z+dz/2],
    ]
    
    # Define the 6 faces
    faces = [
        [vertices[0], vertices[1], vertices[2], vertices[3]],  # bottom
        [vertices[4], vertices[5], vertices[6], vertices[7]],  # top
        [vertices[0], vertices[1], vertices[5], vertices[4]],  # front
        [vertices[2], vertices[3], vertices[7], vertices[6]],  # back
        [vertices[0], vertices[3], vertices[7], vertices[4]],  # left
        [vertices[1], vertices[2], vertices[6], vertices[5]],  # right
    ]
    
    # Add faces to plot
    poly = Poly3DCollection(faces, alpha=alpha, facecolor=color, edgecolor='black', linewidth=linewidth)
    ax.add_collection3d(poly)

In [ ]:
# Create 3D visualization
fig = plt.figure(figsize=(8, 5))
ax = fig.add_subplot(111, projection='3d')

# Plot the track segment as a line
ax.plot([track['x_start'][0], track['x_end'][0]],
        [track['y_start'][0], track['y_end'][0]],
        [track['z_start'][0], track['z_end'][0]],
        'r-', linewidth=3, label='Track segment', zorder=10, alpha=0.8)

# Plot start and end points
ax.scatter([track['x_start'][0]], [track['y_start'][0]], [track['z_start'][0]],
          c='green', s=100, marker='o', label='Start', zorder=11)
ax.scatter([track['x_end'][0]], [track['y_end'][0]], [track['z_end'][0]],
          c='red', s=100, marker='s', label='End', zorder=11)

# Normalize charges for coloring
charge_normalized = (vox_charge_np - vox_charge_np.min()) / (vox_charge_np.max() - vox_charge_np.min())

# Draw each voxel
cmap = plt.cm.viridis
for i in range(len(x_centers)):
    color = cmap(charge_normalized[i])
    draw_voxel(ax, x_centers[i], y_centers[i], z_centers[i],
              vsize[0], vsize[1], vsize[2],
              color=color, alpha=0.4, linewidth=0.8)

# Add colorbar for charge
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vox_charge_np.min(), vmax=vox_charge_np.max()))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.1, shrink=0.6)
cbar.set_label('Charge (electrons)', rotation=270, labelpad=20)

ax.set_xlabel('X (cm)', fontsize=12)
ax.set_ylabel('Y (cm)', fontsize=12)
ax.set_zlabel('Z (cm)', fontsize=12)
ax.set_title('Track Segment and Voxelized Charge Distribution', fontsize=14, pad=20)
ax.legend(loc='upper left', fontsize=10)

ax.set_xlim(tpc_borders[0,0,:])
ax.set_ylim(tpc_borders[0,1,:])
ax.set_zlim(tpc_borders[0,2,:])

# Set aspect ratio
ax.set_box_aspect([1, 1, 1])

plt.tight_layout()
plt.show()

In [ ]:
# Print detailed voxel information
print("Voxel details (sorted by charge):")
print("="*60)
sorted_indices = np.argsort(vox_charge_np)[::-1]
for i in sorted_indices:
    print(f"Voxel {i:2d}: ({x_centers[i]:5.2f}, {y_centers[i]:5.2f}, {z_centers[i]:5.2f}) cm  |  {vox_charge_np[i]:6.1f} e-  |  {100*vox_charge_np[i]/track['n_electrons'][0]:4.1f}%")

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
def rotate(angle):
    ax.view_init(elev=30, azim=angle)
frames = np.linspace(0, 360, 60)
ani = FuncAnimation(fig, rotate, frames=frames, interval=10)
ani.save("voxelized_track.gif", writer=PillowWriter(fps=10))
